In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamedbentalb/lipreading-dataset")

print("Path to dataset files:", path)

100%|██████████| 404M/404M [00:08<00:00, 51.4MB/s] 

Extracting files...


Path to dataset files: C:\Users\6\.cache\kagglehub\datasets\mohamedbentalb\lipreading-dataset\versions\1


In [1]:
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision
import torch_directml
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch.nn.functional as F

Load the Data

In [ ]:
video_dir = r"mohamedbentalb\lipreading-dataset\versions\1\data\s1"
align_dir = r"mohamedbentalbcreate_dl\lipreading-dataset\versions\1\data\alignments\s1"

# Pair each video with its alignment file
video_files = [f for f in os.listdir(video_dir) if f.endswith('.mpg')]
video_align_pairs = []
for vf in video_files:
    base = os.path.splitext(vf)[0]
    align_path = os.path.join(align_dir, base + ".align")
    video_path = os.path.join(video_dir, vf)
    if os.path.exists(align_path):
        video_align_pairs.append((video_path, align_path))

print(f"Found {len(video_align_pairs)} video-align pairs.")

# First split: train vs (val+test)
train_pairs, valtest_pairs = train_test_split(
    video_align_pairs, test_size=0.4, random_state=42
)

# Second split: validation vs test (split valtest in half)
val_pairs, test_pairs = train_test_split(
    valtest_pairs, test_size=0.5, random_state=42
)

print(f"Train set size: {len(train_pairs)}")
print(f"Validation set size: {len(val_pairs)}")
print(f"Test set size: {len(test_pairs)}")

Found 1000 video-align pairs.
Train set size: 600
Validation set size: 200
Test set size: 200


Pre-Processing dataset, extract each frames

In [3]:
# Preprocessing transform
transform = T.Compose([
    T.ToPILImage(),
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize([0.43216, 0.394666, 0.37645], [0.22803, 0.22145, 0.216989])
])

def preprocess_frames(frames):
    processed = [transform(frame) for frame in frames]
    video_tensor = torch.stack(processed)  # (T, C, H, W)
    video_tensor = video_tensor.permute(1, 0, 2, 3)  # (C, T, H, W)
    return video_tensor.unsqueeze(0)  # (1, C, T, H, W)

def parse_align_file(align_path):
    alignments = []
    with open(align_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                start, end, word = parts
                alignments.append((int(start), int(end), word))
    return alignments

def extract_word_frames(video_path, alignments):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    word_frames = []
    for start, end, word in alignments:
        start_idx = int(start / 1000)
        end_idx = int(end / 1000)
        word_seq = frames[start_idx:end_idx]
        if word_seq:
            word_frames.append((word, word_seq))
    return word_frames

Encoder:

In [4]:
class VideoVAE(nn.Module):
    def __init__(self, latent_dim=128, num_classes=10):  # set num_classes to your vocab size
        super().__init__()
        base = torchvision.models.video.r3d_18(pretrained=True)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.encoder_out_dim = 512
        self.fc_mu = nn.Linear(self.encoder_out_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.encoder_out_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, self.encoder_out_dim),
            nn.ReLU(),
            nn.Linear(self.encoder_out_dim, self.encoder_out_dim)
        )
        self.classifier = nn.Linear(latent_dim, num_classes)  # classification head

    def encode(self, x):
        x = self.encoder(x)
        x = x.flatten(1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        logits = self.classifier(mu)  # use mu for classification
        return recon, mu, logvar, logits

Select Device

In [5]:
# device = torch_directml.device()
device = torch.device("cpu")
# device = torch.device("cuda")

Training loop

In [ ]:
latent_dim = 128
num_epochs = 10
learning_rate = 1e-4
beta = 1.0  # KL loss weight

all_words = []
for _, align_path in train_pairs:
    alignments = parse_align_file(align_path)
    for _, _, word in alignments:
        all_words.append(word)

# Fit LabelEncoder on all words
le = LabelEncoder()
le.fit(all_words)
num_classes = len(le.classes_)
print(f"Number of unique classes: {num_classes}")

#There are 53 classes in the dataset

vae = VideoVAE(latent_dim=latent_dim, num_classes=53).to(device)
optimizer = optim.Adam(vae.parameters(), lr=learning_rate)
vae.train()

# --- Training Loop ---
for epoch in range(num_epochs):
    total_loss = 0
    total_recon = 0
    total_kl = 0
    count = 0
    for video_path, align_path in tqdm(train_pairs, desc=f"Epoch {epoch+1}/{num_epochs}"):
        alignments = parse_align_file(align_path)
        word_frames = extract_word_frames(video_path, alignments)
        for word, frames in word_frames:
            video_tensor = preprocess_frames(frames).to(device)
            optimizer.zero_grad()
            recon, mu, logvar, logits = vae(video_tensor)
            # Flatten input for reconstruction loss
            target = vae.encoder(video_tensor).flatten(1)
            # Reconstruction loss (MSE between recon and encoder output)
            recon_loss = nn.MSELoss()(recon, target)
            # KL divergence loss
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)

            word_idx = torch.tensor([le.transform([word])[0]], dtype=torch.long, device=device)
            class_loss = nn.CrossEntropyLoss()(logits, word_idx)

            # Total loss (tune alpha as needed)
            loss = recon_loss + beta * kl_loss + 0.5 * class_loss
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            count += 1
    print(f"Epoch {epoch+1}: Loss={total_loss/count:.4f}, Recon={total_recon/count:.4f}, KL={total_kl/count:.4f}")

# --- Save the trained model ---
torch.save(vae.state_dict(), "vae_trained.pt")
vae.eval()

Batch size loaded, did not work

In [ ]:
def pad_collate(batch):
    # Remove any samples with empty tensors
    batch = [(v, w) for v, w in batch if v.shape[2] > 0]
    if len(batch) == 0:
        return None, None
    videos, words = zip(*batch)
    max_T = max(v.shape[2] for v in videos)
    padded_videos = []
    for v in videos:
        pad_T = max_T - v.shape[2]
        if pad_T > 0:
            v = F.pad(v, (0,0,0,0,0,pad_T))
        padded_videos.append(v)
    video_batch = torch.cat(padded_videos, dim=0)
    return video_batch, list(words)

class WordVideoDataset(Dataset):
    def __init__(self, video_align_pairs):
        self.samples = []
        for video_path, align_path in video_align_pairs:
            alignments = parse_align_file(align_path)
            for start, end, word in alignments:
                self.samples.append((video_path, start, end, word))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        video_path, start, end, word = self.samples[idx]
        # Extract only the frames for this word on-the-fly
        alignments = [(start, end, word)]
        word_frames = extract_word_frames(video_path, alignments)
        frames = word_frames[0][1]  # get frames for this word
        video_tensor = preprocess_frames(frames)
        return video_tensor, word
    
train_dataset = WordVideoDataset(train_pairs)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,      # Set your batch size here
    shuffle=True,
    pin_memory=True,
    collate_fn=pad_collate
) 

latent_dim = 128
num_epochs = 10
learning_rate = 1e-4
beta = 1.0  # KL loss weight

vae = VideoVAE(latent_dim=latent_dim).to(device)
optimizer = optim.Adam(vae.parameters(), lr=learning_rate)
vae.train()

for epoch in range(num_epochs):
    total_loss = 0
    total_recon = 0
    total_kl = 0
    count = 0
    for video_tensor, word in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        video_tensor = video_tensor.to(device)
        optimizer.zero_grad()
        recon, mu, logvar, _ = vae(video_tensor)
        target = vae.encoder(video_tensor).flatten(1)
        recon_loss = nn.MSELoss()(recon, target)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
        loss = recon_loss + beta * kl_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        count += 1
    print(f"Epoch {epoch+1}: Loss={total_loss/count:.4f}, Recon={total_recon/count:.4f}, KL={total_kl/count:.4f}")

Encode the val and test data

In [ ]:
latent_dim = 128
vae = VideoVAE(latent_dim=latent_dim).to(device)
vae.load_state_dict(torch.load("vae_trained.pt", map_location=device))
vae.eval()

all_mu = []
all_logvar = []
all_labels = []

for video_path, align_path in tqdm(train_pairs, desc="Encoding training set"):
    alignments = parse_align_file(align_path)
    word_frames = extract_word_frames(video_path, alignments)
    with torch.no_grad():
        for word, frames in word_frames:
            video_tensor = preprocess_frames(frames).to(device)
            mu, logvar = vae.encode(video_tensor)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.append(word)

# Save or use the encoded features
torch.save(all_mu, 'tr_mu.pt')
torch.save(all_logvar, 'tr_logvar.pt')
torch.save(all_labels, 'tr_labels.pt')

In [ ]:
vae = VideoVAE(latent_dim=latent_dim).to(device)
vae.load_state_dict(torch.load("vae_trained.pt", map_location=device))
vae.eval()

all_mu = []
all_logvar = []
all_labels = []

for video_path, align_path in tqdm(val_pairs, desc="Encoding validation set"):
    alignments = parse_align_file(align_path)
    word_frames = extract_word_frames(video_path, alignments)
    with torch.no_grad():
        for word, frames in word_frames:
            video_tensor = preprocess_frames(frames).to(device)
            mu, logvar = vae.encode(video_tensor)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.append(word)

# Save or use the encoded features
torch.save(all_mu, 'val_mu.pt')
torch.save(all_logvar, 'val_logvar.pt')
torch.save(all_labels, 'val_labels.pt')

In [ ]:
vae = VideoVAE(latent_dim=latent_dim).to(device)
vae.load_state_dict(torch.load("vae_trained.pt", map_location=device))
vae.eval()

all_mu = []
all_logvar = []
all_labels = []

for video_path, align_path in tqdm(test_pairs, desc="Encoding testdata set"):
    alignments = parse_align_file(align_path)
    word_frames = extract_word_frames(video_path, alignments)
    with torch.no_grad():
        for word, frames in word_frames:
            video_tensor = preprocess_frames(frames).to(device)
            mu, logvar = vae.encode(video_tensor)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.append(word)

# Save or use the encoded features
torch.save(all_mu, 'test_mu.pt')
torch.save(all_logvar, 'test_logvar.pt')
torch.save(all_labels, 'test_labels.pt')

Label Classifier

In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch
import numpy as np

# Load training features and labels
train_mu = torch.load('tr_mu.pt')
train_logvar = torch.load('tr_logvar.pt')
train_labels = torch.load('tr_labels.pt')

# Load validation features and labels
val_mu = torch.load('val_mu.pt')
val_logvar = torch.load('val_logvar.pt')
val_labels = torch.load('val_labels.pt')

# Concatenate mu and logvar for both train and val
X_train = np.concatenate([torch.cat(train_mu).numpy(), torch.cat(train_logvar).numpy()], axis=1)
X_val = np.concatenate([torch.cat(val_mu).numpy(), torch.cat(val_logvar).numpy()], axis=1)
y_train = np.array(train_labels)
y_val = np.array(val_labels)

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

# Train MLP classifier
clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500)
clf.fit(X_train, y_train_enc)

# Training accuracy
y_pred_train = clf.predict(X_train)
acc_train = accuracy_score(y_train_enc, y_pred_train)
print(f"MLP Training accuracy: {acc_train:.4f}")

# Validation accuracy
y_pred_val = clf.predict(X_val)
acc_val = accuracy_score(y_val_enc, y_pred_val)
print(f"MLP Validation accuracy: {acc_val:.4f}")

C:\Users\6\AppData\Local\Temp\ipykernel_9700\2647760125.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_mu = torch.load('tr_mu.pt')
C:\Users\6\AppData\Local\Temp\i

MLP Training accuracy: 0.8694
MLP Validation accuracy: 0.8582
